# 02 — Replica builder

The Streamlit UI's **Replica Builder** lets you draw out a digital twin component by component. Under the hood it builds up a Python data structure and then hands it to a TTL generator.

This notebook shows the same path without the UI: we'll add a **retrofit gas boiler** to Building C, generate valid TTL for it, and upload it to the Alpine Village graph.

Prerequisites: complete `00_setup.md` and run `01_ontology_basics.ipynb` at least once (it seeds the sample graph).

## 2.1 Setup

In [ ]:
import os, sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))
os.environ.setdefault("TRIPLESTORE_BACKEND", "fuseki")
os.environ.setdefault("GRAPHDB_URL", "http://localhost:3030")
# Fuseki needs admin auth for writes (upload / SPARQL update). Compose defaults:
os.environ.setdefault("FUSEKI_ADMIN_USER", "admin")
os.environ.setdefault("FUSEKI_ADMIN_PASSWORD", "admin")

from backend.graphdb import GraphDBClient
from backend.replica_builder.utils import generate_attribute_ttl

client = GraphDBClient(token="local", selected_repo="workspace_demo")
SAMPLE_GRAPH = "<https://digicities.info/tutorial/alpine_village>"
PROJECT_NS   = "https://digicities.info/tutorial/alpine_village/"
client.auth_mode

## 2.2 Describe the new component

Every component is a small Python dict describing the class, label, and a list of attributes. Attributes are keyed by name and carry a `type` plus type-specific fields (`value`, `unit`, `currency`, …). This is exactly the shape the UI builds up in `st.session_state.replica_instances`.

In [ ]:
retrofit_boiler = {
    "uri":   f"{PROJECT_NS}Boiler_C",
    "type":  "EnergyConverter",
    "label": "Retrofit gas boiler at Building C",
    "attributes": {
        "NominalPower": {
            "type":  "Physical",
            "value": 15.0,
            "unit":  "KiloW",
        },
        "Efficiency": {
            "type":  "Physical",
            "value": 0.94,
            "unit":  "",   # dimensionless
        },
        "CapEx": {
            "type":     "UnitBasedCost",
            "value":    350.0,
            "unit":     "KiloW",
            "currency": "CHF",
        },
    },
}
retrofit_boiler

## 2.3 Generate the attribute TTL

`generate_attribute_ttl(attr_uri, attr_name, attr_data, component_type)` returns the TTL lines for a single attribute resource. It handles all the type-specific patterns — units, currency, time-series references, curve data points — so you don't have to remember them.

In [ ]:
sample_name = "NominalPower"
sample_data = retrofit_boiler["attributes"][sample_name]
attr_uri = f"{retrofit_boiler['uri']}_{sample_name}"

for line in generate_attribute_ttl(attr_uri, sample_name, sample_data, retrofit_boiler["type"]):
    print(line)

## 2.4 Build the full component TTL

A component needs its own `a dici_onto:<Type>` + `rdfs:label` + `dici_onto:hasAttribute` links to each of its attributes. That's a couple of lines of string-building on top of `generate_attribute_ttl`.

In [ ]:
def component_ttl(component: dict) -> str:
    attr_uris = {
        name: f"{component['uri']}_{name}"
        for name in component["attributes"]
    }

    lines = [
        "@prefix dici_onto: <https://digicities.info/ontology#> .",
        "@prefix qudt:      <http://qudt.org/schema/qudt/> .",
        "@prefix unit:      <http://qudt.org/vocab/unit/> .",
        "@prefix xsd:       <http://www.w3.org/2001/XMLSchema#> .",
        "@prefix dcterms:   <http://purl.org/dc/terms/> .",
        "@prefix rdfs:      <http://www.w3.org/2000/01/rdf-schema#> .",
        "@prefix cur:       <http://qudt.org/vocab/currency/> .",
        "",
    ]

    attr_refs = " ,\n        ".join(f"<{uri}>" for uri in attr_uris.values())
    lines += [
        f"<{component['uri']}> a dici_onto:{component['type']} ;",
        f'    rdfs:label "{component["label"]}"^^xsd:string ;',
        f"    dici_onto:hasAttribute {attr_refs} .",
        "",
    ]

    for name, data in component["attributes"].items():
        lines += generate_attribute_ttl(attr_uris[name], name, data, component["type"])
        lines.append("")

    return "\n".join(lines)

ttl = component_ttl(retrofit_boiler)
print(ttl)

## 2.5 Validate locally before uploading

`rdflib` will tell you immediately if the generated TTL is syntactically invalid. Always do this before pushing to GraphDB — a bad PUT can poison a named graph.

In [ ]:
import rdflib
g = rdflib.Graph()
g.parse(data=ttl, format="turtle")
print(f"Parsed {len(g)} triples locally.")

## 2.6 Upload to the Alpine Village graph

We want to **add** to the existing named graph, not overwrite it, so `replace_existing=False`.

In [ ]:
client.upload_ttl(
    ttl_str=ttl,
    graph_name=SAMPLE_GRAPH,
    replace_existing=False,
)

client.sparql_api_query(f"""
    PREFIX dici_onto: <https://digicities.info/ontology#>
    PREFIX qudt: <http://qudt.org/schema/qudt/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?attr_label ?value ?unit WHERE {{
      GRAPH {SAMPLE_GRAPH} {{
        <{PROJECT_NS}Boiler_C> dici_onto:hasAttribute ?attr .
        ?attr rdfs:label ?attr_label ; qudt:value ?value .
        OPTIONAL {{ ?attr dici_onto:hasUnitLabel ?unit }}
      }}
    }} ORDER BY ?attr_label
""", out_format="df")

## 2.7 Clean up

To roll back this addition (and leave the village in its original state), delete the triples you added using a `DELETE WHERE`:

In [ ]:
client.sparql_update(f"""
    PREFIX dici_onto: <https://digicities.info/ontology#>

    WITH {SAMPLE_GRAPH}
    DELETE {{
      <{PROJECT_NS}Boiler_C> ?p ?o .
      ?attr ?ap ?ao .
    }}
    WHERE {{
      <{PROJECT_NS}Boiler_C> ?p ?o .
      OPTIONAL {{ <{PROJECT_NS}Boiler_C> dici_onto:hasAttribute ?attr . ?attr ?ap ?ao }}
    }}
""")
print("Retrofit boiler removed.")

## What you just did

- Encoded a component as a plain Python dict — the same shape the Streamlit Replica Builder produces
- Used `backend.replica_builder.utils.generate_attribute_ttl` to serialise each attribute
- Validated locally with `rdflib` before pushing
- Added the component to a named graph via the GraphDB client
- Cleaned up with a SPARQL UPDATE

In a real workspace you'd build up dozens of these dicts and batch-upload. The UI's Excel importer (`backend.replica_builder.utils.process_excel_to_ttl`) does exactly that — takes a populated template, walks rows, and produces one big TTL blob for upload. [`09_excel_import.ipynb`](09_excel_import.ipynb) walks through that path end-to-end — the supported attribute types, the three URI modes, and the Reference-sheet citation flow.

Next up: [`03_scenario_builder.ipynb`](03_scenario_builder.ipynb) — fetching the twin back out of GraphDB in the shape the scenario/assumptions engines expect.